In [1]:
# Block 1: Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Check if TVB is properly imported
try:
    # Import TVB-related libraries
    from tvb.simulator.lab import *
    print("TVB successfully imported!")
except ImportError:
    print("ERROR: TVB is not properly installed or cannot be imported")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tvb/datatypes/surfaces.py:60: UserWarning: Geodesic distance module is unavailable; some functionality for surfaces will be unavailable.
  warnings.warn(msg)


TVB successfully imported!


In [8]:
# Block 2: Define functions for TVB simulation
def setup_simulation(conn_strength=0.0, sampling_period=0.1, simulation_length=1000):
    """
    Set up a TVB simulation with specified parameters.
    
    Parameters:
        conn_strength (float): Connectivity scaling factor
        sampling_period (float): Sampling period in ms
        simulation_length (int): Length of simulation in ms
        random_seed (int): Random seed for reproducibility
        
    Returns:
        TVB Simulator object
    """
    print(f"Setting up simulation with conn_strength={conn_strength}, length={simulation_length}ms")
    start_time = time.time()
    
    # Create a simulator
    sim = simulator.Simulator()
    
    # Set the connectivity - default matrix from TVB
    white_matter = connectivity.Connectivity.from_file()
    white_matter.speed = np.array([4.0])
    white_matter.configure()
    sim.connectivity = white_matter
    
    # Set the coupling strength
    sim.coupling = coupling.Linear(a=np.array([conn_strength]))
    
    # Set the model - using Epileptor model for seizure simulation
    sim.model = models.Epileptor()
    
    # Set the integration scheme
    sim.integrator = integrators.HeunStochastic(dt=0.1, noise=noise.Additive(nsig=np.array([0.00001])))
    
    # Set monitors for output
    sim.monitors = [
        monitors.TemporalAverage(period=sampling_period),
        monitors.Bold(period=2000.0)  # BOLD with TR=2s
    ]
    
    # Configure simulator
    sim.configure()
    
    elapsed = time.time() - start_time
    print(f"Simulation setup completed in {elapsed:.2f} seconds")
    
    return sim

def run_simulation(simulator, simulation_length=1000):
    """
    Run the simulation for a specified duration.
    
    Parameters:
        simulator: TVB Simulator object
        simulation_length (int): Length of simulation in ms
        
    Returns:
        Tuple of (time, data) from simulation results
    """
    print(f"Starting simulation run for {simulation_length}ms")
    start_time = time.time()
    
    # Run the simulation
    (tavg_time, tavg_data), (bold_time, bold_data) = simulator.run(simulation_length=simulation_length)
    
    elapsed = time.time() - start_time
    print(f"Simulation run completed in {elapsed:.2f} seconds")
    print(f"Generated data shape: {tavg_data.shape}")
    
    return tavg_time, tavg_data, bold_time, bold_data

In [9]:
# Block 3: Generate or load simulation data
def generate_simulation_data(num_simulations=10, conn_range=(0.0, 0.1)):
    """
    Generate simulation data with varying connectivity strengths.
    
    Parameters:
        num_simulations (int): Number of simulations to run
        conn_range (tuple): Range of connectivity strengths
        
    Returns:
        DataFrame containing simulation features and target labels
    """
    print(f"Generating {num_simulations} simulations...")
    
    all_features = []
    all_targets = {}
    
    conn_values = np.linspace(conn_range[0], conn_range[1], num_simulations)
    
    for i, conn in enumerate(conn_values):
        print(f"\nSimulation {i+1}/{num_simulations} - Connectivity: {conn:.4f}")
        
        # Setup and run simulation
        sim = setup_simulation(conn_strength=conn)
        tavg_time, tavg_data, bold_time, bold_data = run_simulation(sim)
        
        # Extract features from simulation results
        features = extract_features(tavg_data, tavg_time, bold_data, bold_time)
        
        # Determine seizure properties
        has_seizure, onset_time = detect_seizure(tavg_data, tavg_time)
        
        # Store results
        all_features.append(features)
        all_targets[i] = {
            'has_seizure': has_seizure,
            'onset_time': onset_time if has_seizure else np.nan
        }
        
        print(f"  Seizure detected: {has_seizure}, Onset time: {onset_time if has_seizure else 'N/A'}")
    
    # Convert to DataFrames
    features_df = pd.DataFrame(all_features)
    targets_df = pd.DataFrame.from_dict(all_targets, orient='index')
    
    print(f"\nGenerated {len(features_df)} samples with {features_df.shape[1]} features")
    print(f"Features dataframe shape: {features_df.shape}")
    print(f"Targets dataframe shape: {targets_df.shape}")
    
    return features_df, targets_df

def extract_features(tavg_data, tavg_time, bold_data, bold_time):
    """
    Extract features from simulation data.
    
    Parameters:
        tavg_data: TVB temporal average data
        tavg_time: TVB temporal average time points
        bold_data: TVB BOLD data
        bold_time: TVB BOLD time points
        
    Returns:
        Dictionary of features
    """
    # Print shapes to verify data
    print(f"  Extracting features from tavg_data shape: {tavg_data.shape}")
    print(f"  BOLD data shape: {bold_data.shape}")
    
    # Example feature extraction
    features = {}
    
    # Average activity features
    features['mean_activity'] = np.mean(tavg_data)
    features['max_activity'] = np.max(tavg_data)
    features['std_activity'] = np.std(tavg_data)
    
    # Frequency domain features
    for region_idx in range(min(5, tavg_data.shape[1])):  # First 5 regions
        signal = tavg_data[:, region_idx, 0]
        fft_vals = np.abs(np.fft.rfft(signal))
        features[f'peak_freq_region_{region_idx}'] = np.argmax(fft_vals)
        features[f'power_region_{region_idx}'] = np.sum(fft_vals**2)
    
    # BOLD features
    if bold_data.size > 0:
        features['mean_bold'] = np.mean(bold_data)
        features['max_bold'] = np.max(bold_data)
    
    return features

def detect_seizure(tavg_data, tavg_time, threshold=2.0):
    """
    Detect if a seizure occurs in the simulation data.
    
    Parameters:
        tavg_data: TVB temporal average data
        tavg_time: TVB temporal average time points
        threshold: Activity threshold for seizure detection
        
    Returns:
        Tuple of (has_seizure, onset_time)
    """
    # Simple detection: if activity exceeds threshold
    max_activity = np.max(tavg_data)
    has_seizure = max_activity > threshold
    
    onset_time = None
    if has_seizure:
        # Find first time when activity exceeds threshold
        for i, t in enumerate(tavg_time):
            if np.any(tavg_data[i, :, 0] > threshold):
                onset_time = t
                break
    
    return has_seizure, onset_time

In [10]:
# Block 4: Generate or load data (run this block)
# Set to True to generate new data, False to load existing data (if available)
GENERATE_NEW_DATA = True

try:
    if GENERATE_NEW_DATA:
        print("Generating new simulation data...")
        start_time = time.time()
        features, targets = generate_simulation_data(num_simulations=5)  # Reduced for testing
        elapsed = time.time() - start_time
        print(f"Data generation completed in {elapsed:.2f} seconds")
        
        # Save generated data
        results_dir = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        os.makedirs(results_dir, exist_ok=True)
        features.to_csv(os.path.join(results_dir, 'features.csv'), index=False)
        targets.to_csv(os.path.join(results_dir, 'targets.csv'), index=False)
        print(f"Data saved to {results_dir}")
    else:
        # Try to load existing data
        print("Attempting to load existing data...")
        # You would need to specify the correct path to your data here
        results_dir = "results_YYYYMMDD_HHMMSS"  # Replace with actual directory
        features = pd.read_csv(os.path.join(results_dir, 'features.csv'))
        targets = pd.read_csv(os.path.join(results_dir, 'targets.csv'))
        print(f"Loaded data: {features.shape[0]} samples with {features.shape[1]} features")
        
    # Check if data is empty
    if features.empty or targets.empty:
        print("ERROR: Generated/loaded data is empty!")
        print(f"Features shape: {features.shape}")
        print(f"Targets shape: {targets.shape}")
except Exception as e:
    print(f"Error during data generation/loading: {str(e)}")
    import traceback
    traceback.print_exc()

Generating new simulation data...
Generating 5 simulations...

Simulation 1/5 - Connectivity: 0.0000
Setting up simulation with conn_strength=0.0, length=1000ms
2025-03-02 10:15:24,627 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation setup completed in 0.01 seconds
Starting simulation run for 1000ms
Simulation run completed in 1.06 seconds
Generated data shape: (10000, 2, 76, 1)
  Extracting features from tavg_data shape: (10000, 2, 76, 1)
  BOLD data shape: (0,)
  Seizure detected: True, Onset time: 0.05

Simulation 2/5 - Connectivity: 0.0250
Setting up simulation with conn_strength=0.025, length=1000ms
2025-03-02 10:15:25,699 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation setup completed in 0.01 seconds
Starting simulation run for 1000ms
Simulation run completed in 1.04 seconds
Generated data shape: (10000, 2, 76, 1)
  Extracting features from tavg_data shape: (10000, 2, 76, 1)
  BOLD data shape: (0,)
  Seizure detected: